# openEO Earth Engine map (S1 monthly RGB)

Earth Engine backend version of your workflow for Oetztal.

Note: this backend does not support `rename_labels`/`merge_cubes`, so we run three monthly jobs (R/G/B) and combine locally.

In [19]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openeo
from PIL import Image

%matplotlib widget

In [20]:
# -------- user parameters --------
BACKEND_URL = "https://earthengine.openeo.org/v1.0"

# Put credentials here or via env vars OPENEO_EE_USER / OPENEO_EE_PASSWORD
USERNAME = os.getenv("OPENEO_EE_USER", "group11")
PASSWORD = os.getenv("OPENEO_EE_PASSWORD", "test123")

CENTER_LAT = 46.83
CENTER_LON = 10.78
HALF_SIZE_DEG = 0.08

MARCH_START = "2024-03-01"
APRIL_START = "2024-04-01"
MAY_START = "2024-05-01"
JUNE_START = "2024-06-01"

ROOT = Path("/home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis")
DATA_DIR = ROOT / "data" / "oetztal" / "earthengine_rgb"
FIG_DIR = ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

BBOX = {
    "west": CENTER_LON - HALF_SIZE_DEG,
    "south": CENTER_LAT - HALF_SIZE_DEG,
    "east": CENTER_LON + HALF_SIZE_DEG,
    "north": CENTER_LAT + HALF_SIZE_DEG,
}

print("Backend:", BACKEND_URL)
print("BBox:", BBOX)
print("Temporal range:", MARCH_START, "to", JUNE_START)

Backend: https://earthengine.openeo.org/v1.0
BBox: {'west': 10.7, 'south': 46.75, 'east': 10.86, 'north': 46.91}
Temporal range: 2024-03-01 to 2024-06-01


In [21]:
def monthly_png(month_start: str, month_end: str, out_path: Path) -> Path:
    cube = con.load_collection(
        "COPERNICUS/S1_GRD",
        spatial_extent=BBOX,
        temporal_extent=[month_start, month_end],
        bands=["VV"],
        fetch_metadata=False,
    )
    result = cube.mean_time().save_result(format="PNG")
    job = result.create_job(title=f"oetztal-s1-vv-{month_start}")
    job.start_and_wait()

    month_dir = DATA_DIR / month_start
    month_dir.mkdir(parents=True, exist_ok=True)
    job.get_results().download_files(str(month_dir))

    png_files = sorted(month_dir.rglob("*.png"))
    if not png_files:
        raise RuntimeError(f"No PNG downloaded for {month_start} in {month_dir}")

    src = png_files[-1]
    if src != out_path:
        out_path.write_bytes(src.read_bytes())
    return out_path


In [22]:
con = openeo.connect(BACKEND_URL)
try:
    con.authenticate_basic(USERNAME, PASSWORD)
    print("Authenticated with basic auth.")
except Exception as exc:
    raise RuntimeError(
        "Authentication failed on earthengine.openeo.org. "
        "Please verify OPENEO_EE_USER / OPENEO_EE_PASSWORD (or USERNAME/PASSWORD in this notebook)."
    ) from exc

march_png = DATA_DIR / "oetztal_march_vv.png"
april_png = DATA_DIR / "oetztal_april_vv.png"
may_png = DATA_DIR / "oetztal_may_vv.png"

march_png = monthly_png(MARCH_START, APRIL_START, march_png)
april_png = monthly_png(APRIL_START, MAY_START, april_png)
may_png = monthly_png(MAY_START, JUNE_START, may_png)

print("Downloaded:", march_png)
print("Downloaded:", april_png)
print("Downloaded:", may_png)

Failed to parse API error response: [500] '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="utf-8">\n<title>Error</title>\n</head>\n<body>\n<pre>TypeError: res.status is not a function<br> &nbsp; &nbsp;at Server.errorHandler (file:///home/m_mohr08/openeo-earthengine-driver/src/server.js:121:8)<br> &nbsp; &nbsp;at Layer.handleRequest (/home/m_mohr08/openeo-earthengine-driver/node_modules/router/lib/layer.js:152:17)<br> &nbsp; &nbsp;at trimPrefix (/home/m_mohr08/openeo-earthengine-driver/node_modules/router/index.js:342:13)<br> &nbsp; &nbsp;at /home/m_mohr08/openeo-earthengine-driver/node_modules/router/index.js:297:9<br> &nbsp; &nbsp;at processParams (/home/m_mohr08/openeo-earthengine-driver/node_modules/router/index.js:582:12)<br> &nbsp; &nbsp;at next (/home/m_mohr08/openeo-earthengine-driver/node_modules/router/index.js:291:5)<br> &nbsp; &nbsp;at Server.logRequest (file:///home/m_mohr08/openeo-earthengine-driver/src/server.js:214:3)<br> &nbsp; &nbsp;at Layer.handleRequest (/h

RuntimeError: Authentication failed on earthengine.openeo.org. Please verify OPENEO_EE_USER / OPENEO_EE_PASSWORD (or USERNAME/PASSWORD in this notebook).

In [ ]:
def to_unit_gray(path: Path) -> np.ndarray:
    arr = np.asarray(Image.open(path).convert("L"), dtype=np.float32)
    lo = np.nanpercentile(arr, 2)
    hi = np.nanpercentile(arr, 98)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    arr = (arr - lo) / (hi - lo)
    return np.clip(arr, 0, 1)

R = to_unit_gray(march_png)
G = to_unit_gray(april_png)
B = to_unit_gray(may_png)
rgb = np.dstack([R, G, B])

fig, ax = plt.subplots(figsize=(8, 8), facecolor="white")
ax.imshow(rgb)
ax.set_title("Oetztal S1 monthly RGB (Mar/Apr/May)")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()

fig_path = FIG_DIR / "oetztal_s1_monthly_rgb_earthengine.png"
fig.savefig(fig_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved figure:", fig_path)